# Studio 2 — Klasifikasi Sampah · Pelatihan di Google Colab

Deep Learning (IF25-40401) · Teknik Informatika ITERA · Kelompok **15/DUO FF**
(Muhammad Fadhilah Akbar · 123140003 — Muhammad Fajri Firdaus · 123140050)

Notebook ini disetel untuk **EVA-02 Base @448** — `timm` `eva02_base_patch14_448`,
bobot `.mim_in22k_ft_in1k` (pretrain MIM ImageNet-22k → fine-tune ImageNet-1k).
Ini kandidat **naik skala** dari model yang sekarang terpasang (**EVA-02 Small @336**).
Berkas arsitektur + transform: `experiments/eva02_base_448_model.py`.

```
ZIP dataset di Drive  ->  ekstrak & deteksi  ->  repo dari GitHub (kalau belum ada)
    ->  periksa dataset  ->  atur split (dihitung di sini, dari seed)  ->  latih
    ->  uji dengan evaluate.py resmi  ->  grafik  ->  unduh hasil
```

**Janji penting:** sel kode bagian **6–12** adalah **salinan langsung dari `train.py`**
(diambil dari berkasnya, bukan ditulis ulang) — termasuk `build_train_transform()`
yang meneruskan `MEAN`/`STD` dari berkas arsitektur dan
`train_one_epoch(..., clip_grad=...)`. Karena itu hasil di Colab **sama** dengan
`python train.py` di laptop bila konfigurasinya sama.

**Kontrak submission (jangan dilanggar).** `get_model()` pada berkas arsitektur **hanya
membangun bentuk** (`pretrained=False`) dan **tidak mengunduh apa pun**. Yang menyalin
bobot ImageNet adalah **`muat_bobot_pretrained()`** (dipanggil sel LATIH), dan yang
memuat `model.pth` adalah **`evaluate.py`**. Jangan pindahkan pemuatan bobot ke
dalam `get_model()`.

| Sel | Isi | Wajib? |
|---|---|---|
| 1 | Cek GPU | ya, sekali |
| 2 | Ambil dataset (Drive / unggah ZIP) | ya, sekali |
| 3 | Ekstrak, deteksi, + repo dari GitHub | ya, sekali |
| 3b | Cek berkas arsitektur **tanpa GPU / tanpa unduhan** | disarankan |
| 4 | Periksa dataset + contoh gambar | ya, sekali |
| 5 | **KONFIGURASI** — di sinilah yang diubah | ya |
| 6–12 | Impor, konstanta, fungsi (salinan `train.py`) | ya, sekali |
| 13 | Data loader + pembagian data (dihitung di sini) | ya |
| 14 | **LATIH** | ya |
| 15 | Uji dengan `evaluate.py` resmi | ya |
| 16–17 | Grafik kurva & confusion matrix | disarankan |
| 18 | Unduh hasil (bobot > 120 MB otomatis jadi fp16) | ya |
| 19 | **Putaran Final: latih 100% data** (lewat `train.py`) | untuk Final |
| 20 | Opsional: K-fold 5 | opsional |
| 21 | Opsional: bandingkan semua percobaan | opsional |
| 22 | Opsional: strategi fine-tuning (RPS) | opsional |


## 1. Cek GPU

Kalau T4 tidak muncul: menu **Runtime → Change runtime type → T4 GPU**, lalu
jalankan ulang sel ini.


In [ ]:
# Cek GPU. Kalau di bawah "CUDA tersedia : False", berarti belum pakai GPU.
# Tanpa GPU, latihan bisa 20-30x lebih lambat.
import subprocess

print("=== Kartu grafis yang terlihat ===")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv",
               shell=True, check=False)

import torch
print("torch          :", torch.__version__)
print("CUDA tersedia  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
else:
    raise SystemExit(
        "GPU belum aktif!\n"
        "  Buka: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU\n"
        "  lalu jalankan ulang notebook dari awal (Runtime -> Run all)."
    )


## 2. Ambil Dataset (dari ZIP milikmu sendiri)

**Semua bahan datang dari ZIP-mu sendiri. Tidak ada unduhan dari mana pun.**

| Cara | Cocok untuk | Kecepatan |
|---|---|---|
| **A. Google Drive** — disarankan | ZIP dataset, unggah sekali, awet | ekstrak 1–2 menit |
| B. Unggah langsung ke Colab | cepat, tapi hilang saat runtime reset | unggah lewat browser |

**Isi ZIP minimum yang dibutuhkan:** folder `train/` berisi 5 subfolder kelas
(`botol_plastik`, `gelas_plastik`, `kertas_kardus`, `saset_kemasan`, `styrofoam`).

**Kalau mau sekalian berkas repo** (`evaluate.py` dan `submission/model.py`),
sertakan juga di ZIP. Itu **disarankan**, supaya tidak ada bagian yang perlu
diambilkan dari tempat lain.

**Format ZIP apa pun boleh.** Notebook mendeteksi sendiri isinya, tidak perlu
dirapikan dulu.

**Cara A:** unggah ZIP-mu ke Drive di folder `MyDrive/Tugas Studio 2/Dataset`.
**Cara B:** ubah `PAKAI_DRIVE = False` di sel berikut, lalu sel akan meminta ZIP.

In [ ]:
# ============================================================
# PENGAMBILAN BERKAS  (semua dari ZIP milikmu sendiri)
# ============================================================
PAKAI_DRIVE = True                                                  # False = unggah langsung ke Colab
ZIP_DI_DRIVE = "/content/drive/MyDrive/Tugas Studio 2/Dataset"      # folder di Drive yang memuat ZIP
ZIP_REPO_OPSIONAL = None                                            # path ZIP repo, bila terpisah

from pathlib import Path

if PAKAI_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    sumber = Path(ZIP_DI_DRIVE)
    print("Folder Drive:", sumber)
    if sumber.is_dir():
        zips = sorted(sumber.glob("*.zip"))
        for z in zips:
            print("   {:<52}{:8.1f} MB".format(z.name, z.stat().st_size / 1e6))
        if not zips:
            print("   (belum ada berkas .zip di folder itu)")
    else:
        print("   (folder belum ada — buat dulu di Drive, atau ubah ZIP_DI_DRIVE)")
else:
    from google.colab import files
    print("Pilih berkas ZIP dari komputermu (tunggu sampai 100% selesai):")
    unggahan = files.upload()
    sumber = Path("/content")
    for nama in unggahan:
        Path("/content", nama).write_bytes(unggahan[nama])
    print("terunggah:", list(unggahan))

print("")
print("Sel berikutnya mengekstrak & mendeteksi isi ZIP secara otomatis.")

## 3. Ekstrak & Deteksi Otomatis

Sel ini mencari sendiri, **semua di dalam ZIP-mu**:

- **folder data** — folder yang langsung memuat kelima subfolder kelas
  (`botol_plastik`, `gelas_plastik`, `kertas_kardus`, `saset_kemasan`, `styrofoam`)
- **folder repo** — folder yang memuat `evaluate.py` dan `submission/model.py`;
  **kalau tidak ada di ZIP, notebook otomatis clone dari GitHub**

Jadi tidak perlu tahu struktur ZIP-nya.

> **Kalau `submission/model.py` tidak ada di ZIP**, sel ini membuat salinan
> bawaan (ResNet-18 + transform validasi) langsung di dalam Colab — tanpa
> mengunduh apa pun. Itu berkas yang sama dengan contoh di `submission/model.py`
> repo kelompok, jadi hasilnya tetap sesuai ketentuan.

In [ ]:
# ============================================================
# EKSTRAK & DETEKSI DATASET
# ============================================================
import argparse
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# ---- AWAL ARGPARSER ----
# Sel ini TIDAK butuh argumen perintah. Blok di bawah disediakan supaya fungsi
# yang sama bisa dipakai ulang sebagai skrip kalau perlu (semuanya None = abaikan).
_p = argparse.ArgumentParser(add_help=True, description="Latih ResNet-18 Studio 2 di Colab")
_p.add_argument("--data", default=None, help="folder train/ langsung (lewati ekstrak ZIP)")
_p.add_argument("--epochs", type=int, default=None)
_p.add_argument("--val-frac", dest="val_frac", type=float, default=None)
_p.add_argument("--kfolds", type=int, default=None)
_p.add_argument("--fold", type=int, default=None)
_p.add_argument("--seed", type=int, default=None)
_p.add_argument("--batch-size", dest="batch_size", type=int, default=None)
_p.add_argument("--lr", type=float, default=None)
_p.add_argument("--limit", type=int, default=None)
_p.add_argument("--freeze-epochs", dest="freeze_epochs", type=int, default=None)
_p.add_argument("--scratch", action="store_true")
_p.add_argument("--no-class-weight", dest="class_weight", action="store_false")
_p.add_argument("--tag", default=None)
_args, _sisa = _p.parse_known_args()
if _sisa:
    print("[info] argumen tidak dikenal, diabaikan:", _sisa)
# ---- AKHIR ARGPARSER ----

EKSTRAK = Path("/content/kerja")
KELAS = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]


# ---------- pencarian otomatis ----------
def cari_data(akar):
    """Folder yang langsung memuat kelima subfolder kelas."""
    kandidat = []
    for d in Path(akar).rglob("*"):
        if d.is_dir() and set(KELAS) <= {p.name for p in d.iterdir() if p.is_dir()}:
            kandidat.append(d)
    if not kandidat:
        return []
    kandidat.sort(key=lambda d: sum(len(list((d / k).glob("*.jpg"))) for k in KELAS),
                  reverse=True)
    return kandidat


def cari_repo(akar):
    """Folder yang memuat evaluate.py dan submission/model.py."""
    for d in Path(akar).rglob("*"):
        if d.is_dir() and (d / "evaluate.py").is_file() \
                and (d / "submission" / "model.py").is_file():
            return d
    return None


# ---------- tentukan folder data ----------
if _args.data:
    # mode berkas .py: folder data ditunjuk langsung, ZIP tidak diurus
    DATA_DIR = Path(_args.data)
    if not DATA_DIR.is_dir():
        raise SystemExit("[GAGAL] --data bukan folder yang ada: {}".format(DATA_DIR))
    print("[info] DATA_DIR dari --data:", DATA_DIR)
    EKSTRAK.mkdir(parents=True, exist_ok=True)
else:
    # Nilai ini diisi sel "Ambil Dataset" di atas. Kalau sel itu tidak dijalankan,
    # dipakai /content sebagai gantinya.
    try:
        sumber_zip = Path(sumber)
        zip_repo = Path(ZIP_REPO_OPSIONAL) if ZIP_REPO_OPSIONAL else None
    except NameError:
        sumber_zip, zip_repo = Path("/content"), None

    EKSTRAK.mkdir(parents=True, exist_ok=True)
    daftar_zip = sorted(sumber_zip.glob("*.zip"))
    if zip_repo:
        daftar_zip.append(zip_repo)
    if not daftar_zip:
        raise SystemExit(
            "Tidak ada berkas .zip di {}\n"
            "  Di notebook  : periksa sel 2 (Ambil Dataset).\n"
            "  Di berkas .py: taruh ZIP dataset di /content, atau jalankan dengan\n"
            "                 --data <folder train/ yang sudah ada>.".format(sumber_zip))

    for z in daftar_zip:
        tujuan = EKSTRAK / z.stem.replace(" ", "_")[:40]
        if tujuan.exists():
            print("sudah diekstrak:", z.name)
            continue
        print("mengekstrak {} ...".format(z.name))
        tujuan.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(tujuan)

    kandidat = cari_data(EKSTRAK)
    if not kandidat:
        raise SystemExit(
            "Folder kelas tidak ditemukan di dalam ZIP.\n"
            "  yang dicari: {}\n"
            "  Pastikan ZIP memuat folder 'train/' berisi 5 subfolder kelas itu.".format(KELAS))
    DATA_DIR = kandidat[0]
    print("")
    print("folder DATA :", DATA_DIR)
    for d in kandidat[1:]:
        print("   (kandidat lain, diabaikan):", d)

REPO_DIR = cari_repo(EKSTRAK if EKSTRAK.exists() else Path("/content"))
print("folder REPO :", REPO_DIR if REPO_DIR else "(tidak ada di ZIP)")

# ---------- kalau repo tidak ada di ZIP -> ambil dari GitHub ----------
# Di Colab, folder repo tidak perlu ikut ZIP. Cukup clone dari GitHub; isinya
# kecil (kode saja, dataset tidak ikut ke repo).
REPO_GIT = "https://github.com/fajrifirdaus/DL-Studio-Proyek-2-Klasifikasi-Sampah.git"

if REPO_DIR is None:
    REPO_DIR = Path("/content/repo")
    if (REPO_DIR / ".git").is_dir():
        print("folder repo sudah ada, tarik pembaruan terbaru ...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        print("folder repo tidak ada di ZIP -> clone dari GitHub ...")
        hasil_clone = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_GIT, str(REPO_DIR)],
            capture_output=True, text=True)
        if hasil_clone.returncode != 0:
            print(hasil_clone.stderr[-500:])
            raise SystemExit(
                "[GAGAL] clone GitHub gagal.\n"
                "  Periksa: (1) internet hidup, (2) nama repo/URL benar, (3) repo publik.\n"
                "  Alternatif: sertakan folder repo (evaluate.py + submission/model.py)\n"
                "  di dalam ZIP dataset, seperti cara sebelumnya.")
    print("folder REPO (GitHub):", REPO_DIR)
else:
    print("folder REPO (dari ZIP) :", REPO_DIR)

# ---------- salinan bawaan submission/model.py (tanpa unduh apa pun) ----------
# Isinya SAMA dengan contoh di submission/model.py repo kelompok:
# ResNet-18 head 5 kelas + transform validasi 224.
MODEL_PY_BAWAAN = '''"""Contoh submission: ResNet-18 dengan head 5 kelas.

Aturan:
  - get_model() hanya MEMBANGUN arsitektur (weights=None, tanpa mengunduh bobot).
  - get_transform() harus SAMA dengan transform validasi saat melatih.
  - Output: logits [batch, 5], urutan kelas alfabetis.
"""
import torch.nn as nn
from torchvision import models, transforms

NUM_CLASSES = 5


def get_model():
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


def get_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
'''

if not (REPO_DIR / "evaluate.py").is_file():
    print("   PERHATIAN: evaluate.py tidak ada di ZIP.")
    print("   Salin evaluate.py dari repo kelompok ke dalam ZIP, karena skrip itulah")
    print("   yang dipakai asisten dosen untuk menilai. Tanpa itu, langkah uji resmi")
    print("   di sel 15 tidak bisa dijalankan.")

if not (REPO_DIR / "submission" / "model.py").is_file():
    (REPO_DIR / "submission").mkdir(parents=True, exist_ok=True)
    (REPO_DIR / "submission" / "model.py").write_text(MODEL_PY_BAWAAN, encoding="utf-8")
    print("   submission/model.py dibuat dari salinan bawaan (tanpa unduhan).")

# ---------- PENTING: daftarkan folder repo ke sys.path ----------
# Tanpa baris ini, `from submission.model import ...` akan gagal dengan
# "No module named 'submission'", karena Python mencarinya dari /content,
# sedangkan submission/ berada di dalam folder repo (REPO_DIR).
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("")
print("DATA_DIR =", DATA_DIR)
print("REPO_DIR =", REPO_DIR)
print("evaluate.py ada?          ", (REPO_DIR / "evaluate.py").is_file())
print("submission/model.py ada?  ", (REPO_DIR / "submission" / "model.py").is_file())

# Buktikan import-nya sudah bisa (gagal cepat di sini kalau masih salah,
# bukan nanti di tengah latihan)
import submission.model as _sm
print("import submission.model  : OK  (", _sm.__file__, ")")

# ---------- tentukan jumlah gambar (dipakai di sel berikutnya) ----------
jumlah = {k: len([p for p in (DATA_DIR / k).iterdir() if p.suffix.lower() == ".jpg"])
          for k in KELAS}
TOTAL = sum(jumlah.values())

## 3b. Cek Berkas Arsitektur **tanpa GPU & tanpa unduhan**

Sel murah ini menirukan cara `evaluate.py` memuat berkas: import berkas arsitektur →
`get_model()` → `get_transform()`, **tanpa** memanggil `muat_bobot_pretrained()`.
Harapannya **nol unduhan**. Hasil benar untuk EVA-02 Base: **86.352.389** parameter dan
transform **[3, 448, 448]**.

Kalau muncul angka ±22 juta → itu EVA-02 **Small** (berkas/backbone salah). Kalau sel ini
menggantung lalu mengunduh → ada yang salah pada `pretrained=False` di `get_model()`.


In [ ]:
# Uji kontrak TANPA GPU dan TANPA unduhan (menirukan cara evaluate.py memuat berkas).
import importlib.util
from pathlib import Path
from PIL import Image

_berkas = Path(REPO_DIR) / "experiments" / "eva02_base_448_model.py"
if not _berkas.is_file():
    raise SystemExit("[GAGAL] berkas tidak ada: {} - push/unggah dulu ke repo.".format(_berkas))

_spec = importlib.util.spec_from_file_location("cek_arsitektur", _berkas)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

for _f in ("get_model", "get_transform"):
    assert hasattr(_mod, _f), "berkas tidak punya {}()".format(_f)

_model = _mod.get_model()                       # HANYA bentuk - tidak mengunduh bobot
_n = sum(p.numel() for p in _model.parameters())
_tf = _mod.get_transform()
_bentuk = tuple(_tf(Image.new("RGB", (320, 320))).shape)

print("berkas    :", _berkas.name)
print("backbone  :", getattr(_mod, "BACKBONE", "?"))
print("parameter :", "{:,}".format(_n))
print("transform :", _bentuk)
print("muat_bobot_pretrained tersedia:", hasattr(_mod, "muat_bobot_pretrained"),
      "(hanya dipakai saat LATIH)")
assert _bentuk == (3, 448, 448), "transform bukan 448x448 -> periksa IMG_SIZE pada berkas"
assert _n > 80e6, "parameter ~{:.1f} jt - ini bukan EVA-02 Base".format(_n / 1e6)
print("")
print("[OK] get_model() membangun bentuk saja, TANPA unduhan bobot.")


## 4. Periksa Dataset

Menampilkan jumlah gambar per kelas, rasio ketidakseimbangan, dan contoh gambar.
**Cek di sini dulu** sebelum melatih — kalau jumlahnya aneh, berarti folder data
salah terdeteksi.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# jumlah & TOTAL sudah dihitung di sel sebelumnya (dipakai juga oleh KONFIGURASI)

print("{:<18}{:>8}{:>9}".format("kelas", "gambar", "porsi"))
print("-" * 36)
for k in KELAS:
    print("{:<18}{:>8}{:>8.1f}%".format(k, jumlah[k], 100 * jumlah[k] / TOTAL))
print("-" * 36)
print("{:<18}{:>8}".format("TOTAL", TOTAL))
print("")
print("rasio kelas terbesar:terkecil = {:.2f}x".format(max(jumlah.values()) / min(jumlah.values())))
print("  kelas tersedikit:", min(jumlah, key=jumlah.get),
      "— kelas inilah yang dipantau lewat bobot kelas")

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for kol, k in enumerate(KELAS):
    for bar, p in enumerate(sorted((DATA_DIR / k).glob("*.jpg"))[:2]):
        axes[bar][kol].imshow(Image.open(p).convert("RGB"))
        axes[bar][kol].axis("off")
        if bar == 0:
            axes[bar][kol].set_title("{}\n({} gambar)".format(k, jumlah[k]), fontsize=10)
plt.tight_layout()
plt.show()


## 5. KONFIGURASI — **bagian yang diubah**

### Keputusan pembagian data: **85% latih / 15% validasi**

| Opsi | latih | val | val/kelas | Ambang beda nyata* | Catatan |
|---|---:|---:|---:|---:|---|
| **85/15 (dipakai)** | **2.340** | **413** | **83** | **±4,8%** | cepat, data latih maksimal |
| 80/20 | 2.202 | 551 | 110 | ±4,2% | −138 gambar latih demi ketelitian |
| 75/25 | 2.065 | 688 | 138 | ±3,7% | terlalu banyak yang dibuang |
| K-fold 5 (opsional) | 2.204 | 549/fold | 110 | ±4,2% | **tidak ada data terbuang** |

\* **Perkiraan kasar**, bukan uji statistik formal: `1,96 × √(0,25/n)`.
Semakin banyak gambar validasi, semakin kecil ambangnya (semakin teliti).
Rumus yang sama dipakai `experiments/promote.py` supaya angkanya konsisten.

**Kenapa 85/15?** Karena biaya menaikkan ke 20% adalah **138 gambar latih** (perkiraan
dampak ke skor hanya 0,1–0,3 poin, berdasarkan skala log jumlah data), sementara
manfaatnya hanya membuat angka validasi sedikit lebih teliti. Untuk **iterasi cepat**
(banyak percobaan), 15% lebih menguntungkan.

**Kenapa bukan 25%?** Terlalu banyak data latih dibuang untuk perbaikan yang makin kecil.

**Untuk keputusan serius** (memilih 2–3 kandidat terakhir), pakai **K-fold 5**: tidak ada
satu pun gambar terbuang, karena setiap gambar jadi data latih di 4 dari 5 putaran.
Angkanya juga lebih bisa dipegang — diatur lewat `kfolds` di sel ini.

**Untuk submit Final**, latih ulang di **100% data** (tanpa validasi).

Split bersifat **berstrata** (tiap kelas diambil 15% secara proporsional) dan **ber-seed
tetap** (`seed = 42`), jadi pembagiannya sama persis setiap kali dijalankan ulang — tidak
perlu berkas daftar terpisah. Pembagiannya dihitung di sel 13, tepat sebelum latih.

In [ ]:
# ============================================================
# KONFIGURASI — UBAH DI SINI
# ============================================================
# Setelan aktif = resep EVA-02 Base @448 (86,4 jt parameter, 348 MB fp32).
# Resep ini SENGAJA disamakan dengan run EVA-02 Small @336 yang sudah terpasang,
# supaya yang berubah hanya ukuran/resolusi model -- bukan resepnya:
#     AdamW lr 2e-5, wd 0,05, warmup 1 epoch + cosine, clip-grad 1,0, 15 epoch,
#     early stop patience 5 mulai epoch 6, label smoothing 0,1, bobot kelas aktif.
CFG = {
    # ---------- pembagian data ----------
    "val_frac":   0.15,      # 0.15 = 85% latih / 15% validasi (2.340 / 413 gambar)
    "kfolds":     0,         # 0 = split tunggal; 5 = K-fold 5
    "fold":       1,         # lipatan ke berapa (kalau kfolds = 5)
    "seed":       42,        # angka acak tetap -> hasil bisa diulang

    # ---------- pelatihan (resep EVA-02: ViT -> lr kecil, 15 epoch) ----------
    "epochs":            15,
    "lr":                2e-5,
    "warmup":            1,        # 1 epoch menaikkan LR perlahan (resep EVA-02)
    "eta_min":           1e-6,     # LR paling kecil di ujung cosine
    "weight_decay":      0.05,     # resep ViT/ConvNeXt (bukan 1e-4 ala ResNet)
    "label_smoothing":   0.1,
    # 448 px: batch diturunkan OTOMATIS ke 6 oleh sel 13 (224->32, 384->8, 448->6).
    "batch_size":        6,
    # Gradien dipotong pada norma 1,0. WAJIB untuk resep ini (keluarga ViT/ConvNeXt
    # dengan batch kecil; tanpa ini pelatihan bisa runtuh ke tebakan acak).
    "clip_grad":         1.0,
    "patience":          5,        # early stopping: berhenti bila 5 epoch tak membaik
    "start_epoch":       6,        # early stopping baru aktif setelah epoch ini
    "min_delta":         0.001,    # kenaikan minimum agar dianggap MAJU
    "freeze_epochs":     0,        # 0 = full fine-tuning sejak epoch 1 (resep terpasang)
    "class_weight":      True,     # bobot kelas N/(C x n_c) -> menjaga kertas_kardus
    "scratch":           False,    # True = tanpa bobot ImageNet (CNN from scratch)

    # ---------- arsitektur ----------
    # "arsitektur" = berkas .py di experiments/ berisi get_model() + get_transform()
    # milik sendiri. None = pakai submission/model.py (EVA-02 Small @336).
    # Kandidat siap pakai:
    #     "experiments/eva02_base_448_model.py"    + backbone "eva02_base"   <- setelan sekarang
    #     "experiments/eva02_small_model.py"       + backbone "eva02_small"  (yang terpasang)
    #     "experiments/efficientnet_v2s_model.py"  + backbone "efficientnet_v2_s"
    #     "experiments/resnet34_model.py"          + backbone "resnet34"
    "arsitektur":  "experiments/eva02_base_448_model.py",
    "backbone":    "eva02_base",   # label di metrics.json; berkas arsitektur yang menyalin bobot

    # ---------- kunci eksperimen ----------
    # False = percobaan: bobot masuk runs/<tag>/<tag>.pth (submission/ tidak tersentuh)
    # True  = putaran Final: bobot masuk submission/model.pth (yang dinilai)
    "tulis_submission": False,

    # ---------- lain-lain ----------
    "num_workers": 2,              # Colab = Linux, boleh 2
    "limit":       0,              # >0 = pakai sebagian data saja (uji cepat)
    "tag":         "eva02_base_448_lr2e-5_seed42",
}

# ---------- timpa CFG dengan argumen perintah (kalau ada) ----------
# _args berasal dari sel "Ekstrak & Deteksi" (diabaikan di notebook: semuanya None)
for _nama in ["epochs", "val_frac", "kfolds", "fold", "seed", "batch_size",
              "lr", "limit", "freeze_epochs", "tag", "arsitektur", "backbone"]:
    _nilai = getattr(_args, _nama, None)
    if _nilai is not None:
        CFG[_nama] = _nilai
if _args.scratch:
    CFG["scratch"] = True
if _args.class_weight is False:
    CFG["class_weight"] = False

# tag bawaan: JANGAN None -> percobaan ber-limit tidak tercampur dengan yang penuh
CFG["tag"] = CFG["tag"] or (
    "{}_".format(CFG["backbone"])
    + ("kf{}".format(CFG["fold"]) if CFG["kfolds"] else "split")
    + "_seed{}".format(CFG["seed"])
)

print("KONFIGURASI")
for k, v in CFG.items():
    print("  {:<16} {}".format(k, v))

print("")
print("perkiraan: latih {} gambar | validasi {} gambar".format(
    TOTAL - int(TOTAL * CFG["val_frac"]), int(TOTAL * CFG["val_frac"])))
print("")
print("CATATAN: kalau CFG['limit'] > 0 (uji cepat), my_val/ TIDAK ditulis ulang")
print("        supaya alat ukur 413 gambar tidak rusak oleh split ber-limit.")


## 5b. Cara pakai sel ini — ganti backbone, tag, dan kunci lain

**Hanya sel 5 ini yang perlu diubah.** Setelah mengubahnya, **jalankan ulang sel 13 dan
sel 14**. Mengubah sel 5 tanpa menjalankan ulang sel 13–14 **tidak mengubah apa pun**.

### a) Resep siap tempel

```python
# EVA-02 Base @448 — setelan yang aktif sekarang (kandidat naik skala)
CFG["arsitektur"] = "experiments/eva02_base_448_model.py"
CFG["backbone"]   = "eva02_base"
CFG["epochs"] = 15;  CFG["lr"] = 2e-5;  CFG["warmup"] = 1
CFG["weight_decay"] = 0.05;  CFG["clip_grad"] = 1.0
CFG["batch_size"] = 6;  CFG["patience"] = 5;  CFG["start_epoch"] = 6
CFG["tag"] = "eva02_base_448_lr2e-5_seed42"

# EVA-02 Small @336 — model yang sekarang terpasang di submission/
CFG["arsitektur"] = "experiments/eva02_small_model.py"
CFG["backbone"] = "eva02_small";  CFG["batch_size"] = 8
CFG["tag"] = "eva02_small_336_lr2e-5_seed42"

# EfficientNet V2-S @384 — backbone Preview 1
CFG["arsitektur"] = "experiments/efficientnet_v2s_model.py"
CFG["backbone"] = "efficientnet_v2_s";  CFG["batch_size"] = 8;  CFG["tag"] = "coba_v2s"

# Kembali ke ResNet-18 @224 (contoh starter kit)
CFG["arsitektur"] = None;  CFG["backbone"] = "resnet18";  CFG["tag"] = "coba_resnet18"
```

### b) Empat aturan yang tidak boleh dilanggar

| # | Aturan | Kalau dilanggar |
|---|---|---|
| 1 | `arsitektur` dan `backbone` **wajib sepasang** (EVA-02 Base ↔ `eva02_base`, EVA-02 Small ↔ `eva02_small`, V2-S ↔ `efficientnet_v2_s`) | bobot tidak tersalin, model dilatih hampir dari nol (untuk timm malah gagal cepat) |
| 2 | `get_model()` **hanya membangun bentuk** (`pretrained=False`) — tanpa unduhan. Penyalin bobot ImageNet: `muat_bobot_pretrained()`; pemuat `model.pth`: `evaluate.py` | unduhan tak sengaja saat penilaian, menabrak aturan |
| 3 | `tag` **wajib berbeda** tiap percobaan | isi `runs/<tag>/` saling menimpa, angka pembandingnya hilang |
| 4 | Resolusi `get_transform()` harus cocok dengan latih, dan `clip_grad` mengikuti resep kandidat | akurasi turun **tanpa pesan error**, atau pelatihan runtuh (tanpa clip pada batch kecil) |

### c) Kunci `CFG` yang sering dipakai

| Kunci | Nilai | Artinya |
|---|---|---|
| `tulis_submission` | `False` (bawaan) | bobot percobaan → `runs/<tag>/<tag>.pth`; `submission/` tidak tersentuh |
| `tulis_submission` | `True` | bobot → `submission/model.pth` (hanya putaran **Final**) |
| `clip_grad` | `1.0` | potong norma gradien (resep EVA-02, batch kecil) |
| `scratch` | `True` | tanpa bobot ImageNet (CNN from scratch) |
| `class_weight` | `False` | ablasi: matikan pembobotan kelas |
| `freeze_epochs` | `5` | linear probing 5 epoch, lalu seluruh backbone dibuka |
| `kfolds` | `5` | K-fold; jalankan `fold` 1…5 |
| `limit` | `500` | uji cepat — `my_val/` **tidak** ditulis ulang saat limit aktif |

> **Penting:** `freeze_epochs = 0` (full fine-tuning sejak epoch 1) adalah resep yang
> menghasilkan angka di `runs/` sekarang.

### d) Periksa cetakan `KONFIGURASI` sebelum melatih

Kalau tercetak `backbone = resnet18` padahal kamu mau EVA-02 Base, berarti notebook memakai
**fallback bawaan** — biasanya karena repo tidak ditemukan di ZIP dan `git clone` gagal
(lihat cetakan sel 3: `folder REPO` harus menunjuk `/content/repo`). Perbaiki dulu.

Kalau muncul `[GAGAL] berkas arsitektur tidak ada` → berkasnya belum ada di repo/Colab.
Untuk EVA-02 Base: `experiments/eva02_base_448_model.py`.

### e) Setelah sel 14 selesai

Angka untuk README ada di `runs/<tag>/metrics.json`; bobotnya di `runs/<tag>/<tag>.pth`
(nama tag muncul **dua kali**). Sel 15 menguji bobot percobaan ini dengan `evaluate.py`
resmi (**bukan** `submission/`), jadi aman dijalankan berkali-kali. Kalau kandidat menang,
promosikan lewat `experiments/promote.py` di laptop, lalu untuk putaran Final jalankan
**sel 19** (latih 100% data lewat `train.py`).


## 6. Impor & Konstanta

Sel ini dan sel berikutnya (impor dan konstanta) adalah **salinan langsung dari
`train.py`** — diambil otomatis dari berkasnya, bukan ditulis ulang. Itulah alasan hasil
di Colab sama dengan hasil `python train.py` di laptop.

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from submission.model import get_model, get_transform

# ============================================================
# KONFIGURASI DASAR
# ============================================================
# Urutan kelas WAJIB sama dengan evaluate.py dan ImageFolder (alfabetis).


In [ ]:
CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus",
           "saset_kemasan", "styrofoam"]
NUM_CLASSES = len(CLASSES)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


# ============================================================
# UTILITAS UMUM
# ============================================================


## 7. Fungsi Dasar (seed, perangkat, augmentasi)

Augmentasi latih: **crop 0,60–1,00** (sama dengan acuan yang terbukti tembus 0,9373),
flip horizontal, **ColorJitter 0,3/0,3/0,3/0,05**, dan **RandomErasing p=0,25**.

> **Dua hal yang DIBUANG** dari versi sebelumnya, karena tidak terbukti menolong:
> 1. **Rotasi ±15°** — menambah waktu latih tanpa bukti menaikkan macro-F1; acuan
>    yang tembus 0,9373 tidak memakainya.
> 2. **Crop sempit (0,70–1,00) + ratio khusus** — rentang 0,60–1,00 membuat model
>    lebih suka pada bentuk objek, bukan latar belakang. Pada dataset kecil
>    (2.753 gambar) ini menolong.

In [ ]:
# ============================================================
# UTILITAS UMUM  (salinan langsung dari train.py)
# ============================================================
def seed_everything(seed: int) -> None:
    """Tetapkan seed di semua sumber acak supaya hasil bisa diulang."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    """Pilih perangkat: GPU (cuda) > Apple (mps) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def build_train_transform(img_size: int = IMG_SIZE, mean=None, std=None) -> transforms.Compose:
    """Augmentasi untuk data latih pada resolusi `img_size`.

    RESEP INI SENGAJA DISAMAKAN dengan acuan yang sudah terbukti pada dataset
    yang sama (ResNet-18, 15 epoch -> macro-F1 0,9373). Tiga hal yang DIBUANG
    dari versi sebelumnya karena tidak terbukti menolong:

      - RandomResizedCrop: skala 0.60-1.00 (versi lama 0.70-1.00). Rentang lebih
        lebar -> model tidak sempat menghafal latar belakang, dan pada dataset
        kecil (2.753 gambar) itu menolong. Argumen `ratio` juga dibuang karena
        acuan memakai bawaan.
      - RandomRotation DIBUANG. Menambah waktu latih tanpa bukti menaikkan
        macro-F1; acuan yang tembus 0,9373 tidak memakainya.
      - ColorJitter dinaikkan ke 0.3/0.3/0.3/0.05 dan RandomErasing ke p=0.25
        (versi lama 0.3/0.3/0.2/0.02 dan p=0.20) supaya setara acuan.

    RandomHorizontalFlip DIPERTAHANKAN: kelima kelas (botol, gelas, kertas,
    saset, styrofoam) tidak punya ciri yang berubah makna bila dicerminkan.
    Ciri pembeda "berleher botol" vs "tanpa leher" tetap berlaku setelah
    dicerminkan. (Beda dengan dataset isyarat tangan, di mana flip merusak.)
    """
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.60, 1.00)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        transforms.ToTensor(),
        # mean/std mengikuti berkas arsitektur bila disediakan (mis. EVA-02 memakai
        # normalisasi CLIP); bawaan tetap ImageNet -> run lama tidak berubah.
        transforms.Normalize(mean or IMAGENET_MEAN, std or IMAGENET_STD),
        transforms.RandomErasing(p=0.25),
    ])


## 8. Baca Dataset & Buat Split

Split dihitung **dari `seed` tetap** dan **berstrata per kelas** (tiap kelas diambil 15%),
sehingga bisa diulang persis di Colab maupun di laptop.

> **Jujur soal perbedaan dengan baseline:** `train_baseline.py` SEBENARNYA **juga**
> berstrata per kelas — ia mengambil `int(n × 0,15)` tiap kelas. Bedanya hanya
> pembulatan: kita memakai `max(1, round(n × 0,15))`, jadi tiap kelas dapat 15%
> lebih tepat (**413 vs 410** gambar validasi, selisih **3 gambar**). Ini **bukan**
> keunggulan besar, jadi jangan diklaim sebagai perbaikan utama.
>
> Yang benar-benar membedakan ada di tempat lain: **seleksi model pakai macro-F1**
> (bukan akurasi), **pembobotan kelas**, **30 epoch** (baseline 8), dan
> **early stopping**. Lihat tabel "Celah yang Diperbaiki" di README.

In [ ]:
# ============================================================
# MEMBACA DATASET DAN MEMBUAT SPLIT
# ============================================================
def list_dataset(data_dir: Path) -> list:
    """Baca folder <data_dir>/<nama_kelas>/*.jpg -> list berisi list (path, label).

    Urutan kelas mengikuti CLASSES (alfabetis), sama seperti ImageFolder.
    """
    data_dir = Path(data_dir)
    if not data_dir.is_dir():
        raise SystemExit(f"[GAGAL] folder data tidak ditemukan: {data_dir}")

    ada = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
    if set(ada) != set(CLASSES):
        raise SystemExit(
            "[GAGAL] nama folder kelas tidak sesuai.\n"
            f"  diharapkan : {CLASSES}\n"
            f"  ditemukan  : {ada}\n"
            "Pastikan --data menunjuk ke folder 'train/' dari dataset."
        )

    per_kelas = []
    for label, nama in enumerate(CLASSES):
        files = sorted(p for p in (data_dir / nama).iterdir()
                       if p.suffix.lower() in EXTS)
        if not files:
            raise SystemExit(f"[GAGAL] tidak ada gambar di kelas {nama}")
        per_kelas.append([(p, label) for p in files])
    return per_kelas


def limit_per_class(per_kelas: list, limit: int) -> list:
    """Ambil sebagian data saja (untuk uji cepat). limit = total gambar."""
    if not limit:
        return per_kelas
    per = max(1, limit // len(per_kelas))
    return [items[:per] for items in per_kelas]


def split_single(per_kelas: list, val_frac: float, seed: int) -> tuple:
    """Split tunggal BERSTRATA: tiap kelas diambil val_frac untuk validasi.

    Return: (train_items, val_items) - keduanya list (path, label).
    """
    rng = random.Random(seed)
    tr, va = [], []
    for items in per_kelas:
        items = list(items)
        rng.shuffle(items)
        n_va = max(1, int(round(len(items) * val_frac)))
        va += items[:n_va]
        tr += items[n_va:]
    return tr, va


def split_kfold(per_kelas: list, kfolds: int, fold: int, seed: int) -> tuple:
    """Split K-fold BERSTRATA. fold dimulai dari 1.

    Tiap kelas dibagi rata menjadi kfolds bagian; bagian ke-`fold` menjadi
    validasi, sisanya latih. Union semua fold validasi = seluruh dataset.
    """
    if not 1 <= fold <= kfolds:
        raise SystemExit(f"[GAGAL] --fold harus 1..{kfolds}, diberikan {fold}")
    rng = random.Random(seed)
    tr, va = [], []
    for items in per_kelas:
        items = list(items)
        rng.shuffle(items)
        for i, item in enumerate(items):
            (va if i % kfolds == fold - 1 else tr).append(item)
    return tr, va


def tulis_my_val(val_items: list, out_dir: Path, meta: dict) -> None:
    """Salin gambar validasi ke folder my_val/ supaya bisa diuji evaluate.py.

    Kalau penanda .split.json sudah ada dan isinya sama, penyalinan dilewati.
    Kalau berbeda (seed/frac berubah), folder dibangun ulang - ini yang
    menutup celah 'split kedaluwarsa' pada baseline.
    """
    out_dir = Path(out_dir)
    marker = out_dir / ".split.json"

    if marker.exists():
        try:
            lama = json.loads(marker.read_text())
        except Exception:
            lama = None
        if lama == meta:
            print(f"[split] my_val/ sudah sesuai (dilewati): {out_dir}")
            return
        print("[split] penanda my_val/ berbeda -> membangun ulang")

    if out_dir.exists():
        shutil.rmtree(out_dir)
    for path, label in val_items:
        d = out_dir / CLASSES[label]
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, d / Path(path).name)
    out_dir.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps(meta, indent=2))
    print(f"[split] my_val/ dibuat: {out_dir} ({len(val_items)} gambar)")


# ============================================================
# DATASET DARI DAFTAR PATH
# ============================================================
class PathListDataset(Dataset):
    """Dataset sederhana dari daftar (path, label) + transform."""

    def __init__(self, items: list, tf):
        self.items = items
        self.tf = tf

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        path, label = self.items[idx]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label

## 9. Metrik (macro-F1) & Bobot Kelas

Rumus `macro_f1()` di sini **identik** dengan `metrics()` di `evaluate.py` milik asisten —
sudah diperiksa pada 300 kasus dengan beda maksimum 2,2e-16. Jadi angka di notebook bisa
dibandingkan langsung dengan skor resmi.

Bobot kelas `N / (C × n_c)` membuat `kertas_kardus` (kelas tersedikit) tidak diabaikan.
Catatan rumus: yang bernilai **1,0** adalah rata-rata **tertimbang frekuensi**, bukan
rata-rata biasa antar kelas (yang memang 1,051 untuk data yang tidak seimbang).


In [ ]:
# ============================================================
# METRIK (implementasi harus SAMA dengan evaluate.py milik asisten)  (salinan train.py)
# ============================================================
def macro_f1(y_true: list, y_pred: list, n_classes: int = NUM_CLASSES) -> tuple:
    """Hitung macro-F1, akurasi, confusion matrix, dan F1 per kelas.

    Rumusnya disamakan persis dengan evaluate.py bawaan (tanpa sklearn, karena
    paket dasar asisten hanya torch/torchvision/timm/pillow/numpy).
    """
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    f1_list = []
    for c in range(n_classes):
        tp = cm[c, c]
        fp = int(cm[:, c].sum()) - tp
        fn = int(cm[c, :].sum()) - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1_list.append(2 * prec * rec / (prec + rec) if (prec + rec) else 0.0)

    acc = int(np.trace(cm)) / max(1, len(y_true))
    return float(np.mean(f1_list)), float(acc), cm, f1_list


def class_weight_tensor(labels: list, device) -> tuple:
    """Bobot kelas = N / (C * n_c). Dihitung HANYA dari data latih.

    Catatan penting (sering salah): yang sama dengan 1,0 adalah **rata-rata
    tertimbang frekuensi**, bukan rata-rata biasa:

        sum_c (n_c / N) * w_c = 1,0     <- SELALU benar
        mean_c (w_c)          = 1,0     <- HANYA benar kalau kelas seimbang

    Untuk dataset ini (rasio kelas 1,84x) rata-rata biasa = 1,051.
    Kalau rata-rata biasa dipakai sebagai pemeriksaan "harus 1,0", hasilnya
    akan terlihat seperti kesalahan padahal rumusnya sudah benar.
    """
    counts = np.bincount(np.asarray(labels), minlength=NUM_CLASSES).astype(np.float64)
    counts_aman = np.where(counts == 0, 1.0, counts)
    w = counts.sum() / (NUM_CLASSES * counts_aman)
    return torch.tensor(w, dtype=torch.float32, device=device), counts


## 10. Model, Salin Bobot ImageNet & Strategi Freezing

Bobot disalin hanya ke tensor yang **bentuknya cocok**, lalu jumlahnya dilaporkan — jadi
langsung kelihatan kalau ada yang salah. Strategi *freezing / linear probing / full
fine-tuning* yang diminta RPS diatur lewat `freeze_epochs` di sel 5.


In [ ]:
# ============================================================
# MODEL: SALIN BOBOT IMAGENET + STRATEGI FREEZING  (salinan langsung dari train.py)
# ============================================================
def head_param_ids(model) -> set:
    """Kumpulkan id parameter kepala (fc / classifier) untuk keperluan freezing."""
    # "head" = kepala model timm (mis. ConvNeXt V2: head.norm + head.fc)
    for nama in ("fc", "classifier", "head"):
        head = getattr(model, nama, None)
        if head is not None:
            return {id(p) for p in head.parameters()}
    return set()


def set_backbone_trainable(model, trainable: bool) -> list:
    """Buka/tutup gradien backbone; kepala (head) selalu dilatih.

    PENTING - mengembalikan daftar parameter yang BARU dibuka. Parameter itu
    WAJIB dimasukkan ke optimizer lewat `opt.add_param_group(...)`. Tanpa itu,
    gradiennya dihitung tetapi dibuang, sehingga backbone seolah "dibuka" tapi
    tidak pernah dilatih. (Ini bug yang membuat akurasi mentok: optimizer
    terlanjur dibuat hanya dengan kepala.)
    """
    head_ids = head_param_ids(model)
    baru = []
    for p in model.parameters():
        mau = bool(trainable) or (id(p) in head_ids)
        if mau and not p.requires_grad:
            baru.append(p)
        p.requires_grad = mau
    return baru


def salin_bobot_imagenet(model, backbone: str) -> dict:
    """Salin bobot ImageNet hanya untuk tensor yang bentuknya cocok.

    Ini menggantikan cara baseline yang meng-hardcode awalan "fc." (khusus
    ResNet). Pendekatan berbasis bentuk tensor aman untuk backbone apa pun,
    dan melaporkan berapa tensor yang gagal disalin.
    """
    bobot = models.get_model_weights(backbone)["IMAGENET1K_V1"]
    try:
        src = models.get_model(backbone, weights=bobot, progress=True)
    except Exception as e:
        raise SystemExit(
            "[GAGAL] bobot ImageNet tidak bisa diunduh.\n"
            f"  sebab   : {type(e).__name__}: {e}\n"
            "  solusi  : (1) pastikan internet hidup, atau\n"
            "            (2) unduh bobot sekali di mesin lain lalu salin ke\n"
            "                ~/.cache/torch/hub/checkpoints/, atau\n"
            "            (3) jalankan dengan --scratch (tanpa bobot pretrained)."
        )
    pre, own = src.state_dict(), model.state_dict()

    transfer, dilewati = {}, []
    for k, v in pre.items():
        if k in own and tuple(own[k].shape) == tuple(v.shape):
            transfer[k] = v
        else:
            dilewati.append(k)

    model.load_state_dict(transfer, strict=False)
    return {"backbone": backbone, "disalin": len(transfer),
            "dilewati": dilewati, "total_kunci_model_kita": len(own)}


# ============================================================
# TAMBAHAN NOTEBOOK (tidak ada di train.py): pilih pemuat bobot otomatis
# ============================================================
def ambil_pemuat_bobot(arsitektur) -> callable:
    """Pilih pemuat bobot pretrained: milik berkas arsitektur, atau jalur torchvision.

    train.py melakukan pilihan ini di dalam loop utamanya
    (`pemuat = getattr(modul_arsitektur, "muat_bobot_pretrained", None)`);
    di notebook pilihannya ditaruh di fungsi kecil ini supaya sel 13 dan sel 14
    memakai aturan yang sama.

    Berkas kandidat timm (EVA-02 Base/Small, ConvNeXt V2) menyediakan
    `muat_bobot_pretrained(model)`. Kalau tidak ada, jalur lama (torchvision
    IMAGENET1K_V1) dipakai seperti sebelumnya.

    PENTING: fungsi ini HANYA dipakai saat melatih. `get_model()` tetap hanya
    membangun bentuk (pretrained=False) dan tidak mengunduh apa pun; `evaluate.py`
    juga tidak pernah memanggil pemuat ini.
    """
    if arsitektur is not None:
        pemuat = getattr(arsitektur, "muat_bobot_pretrained", None)
        if pemuat is not None:
            return pemuat

    def _torchvision(model):
        return salin_bobot_imagenet(model, CFG["backbone"])

    return _torchvision


## 11. Jadwal Learning Rate (warmup + cosine)


In [ ]:
# ============================================================
# JADWAL LEARNING RATE (warmup + cosine)  (salinan langsung dari train.py)
# ============================================================
def lr_schedule(epoch: int, warmup: int, base_lr: float, eta_min: float,
                total: int) -> float:
    """LR untuk epoch ke-`epoch` (1-indexed): warmup linear lalu cosine turun."""
    if warmup and epoch <= warmup:
        return base_lr * epoch / float(warmup)
    prog = (epoch - warmup) / float(max(1, total - warmup))
    prog = min(1.0, max(0.0, prog))
    return eta_min + (base_lr - eta_min) * 0.5 * (1.0 + math.cos(math.pi * prog))


## 12. Satu Epoch Latih & Evaluasi


In [ ]:
# ============================================================
# SATU EPOCH LATIH & EVALUASI  (salinan langsung dari train.py)
# ============================================================
def train_one_epoch(model, loader, loss_fn, opt, device, clip_grad: float = 0.0) -> tuple:
    """Latih satu epoch. Return (loss rata-rata, akurasi latih).

    clip_grad > 0: batasi norma gradien (dibutuhkan ConvNeXt V2 dengan batch kecil;
    tanpa ini pelatihan bisa runtuh ke tebakan acak di tengah jalan).
    """
    model.train()
    total_loss, benar, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # set_to_none=True lebih hemat memori daripada mengisi gradien dengan nol
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        if clip_grad > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        opt.step()
        total_loss += loss.item() * y.size(0)
        benar += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / max(1, total), benar / max(1, total)


def evaluate(model, loader, loss_fn, device) -> dict:
    """Evaluasi model: loss, akurasi, macro-F1, F1 per kelas, confusion matrix.

    WAJIB memakai @torch.no_grad(). Tanpa itu, PyTorch menyimpan seluruh grafik
    perhitungan untuk SETIAP batch validasi (13 batch pada split 85/15), dan
    memori GPU menumpuk sampai kehabisan (CUDA out of memory).
    """
    model.eval()
    total_loss, y_true, y_pred = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * y.size(0)
        y_pred += logits.argmax(1).cpu().tolist()
        y_true += y.cpu().tolist()

    f1, acc, cm, per_f1 = macro_f1(y_true, y_pred)
    return {"loss": total_loss / max(1, len(y_true)), "acc": acc,
            "macro_f1": f1, "per_class_f1": per_f1, "cm": cm,
            "n": len(y_true)}


## 13. Siapkan Data Loader & Periksa Kontrak Transform

Sel ini juga **memeriksa kontrak**: resolusi `get_transform()` dari `submission/model.py`
harus sama dengan resolusi latih. Kalau tidak sama, akurasi akan turun **tanpa pesan
error** — jadi lebih baik gagal cepat di sini.

Sel ini juga **menghitung pembagian data** (train/validasi berstrata dari `seed`) dan
menulis folder `my_val/` — 413 gambar pada `val_frac = 0.15` — yang dibutuhkan
`evaluate.py`. Pembagiannya tidak dibaca dari berkas, jadi tidak ada berkas split yang
perlu disiapkan atau di-commit.

In [ ]:
# ============================================================
# DATA LOADER + PERIKSA KONTRAK TRANSFORM
# ============================================================
import numpy as np
from torch.utils.data import DataLoader

per_kelas = limit_per_class(list_dataset(DATA_DIR), CFG["limit"])

# Pembagian data dihitung DI SINI SAJA (tidak pakai berkas split).
# Seed tetap -> pembagiannya sama persis setiap kali dijalankan ulang.
if CFG["kfolds"]:
    train_items, val_items = split_kfold(per_kelas, CFG["kfolds"], CFG["fold"], CFG["seed"])
    print("[split] K-fold {}, fold {} (berstrata)".format(CFG["kfolds"], CFG["fold"]))
else:
    train_items, val_items = split_single(per_kelas, CFG["val_frac"], CFG["seed"])
    print("[split] split tunggal berstrata {:.0%} (seed {})".format(
        CFG["val_frac"], CFG["seed"]))
    print("  pembagian dihitung ulang di sini - seed {} dipakai supaya hasilnya "
          "bisa diulang persis".format(CFG["seed"]))

print("  latih {} gambar | validasi {} gambar".format(len(train_items), len(val_items)))

n_tr = np.bincount([l for _, l in train_items], minlength=NUM_CLASSES)
n_va = np.bincount([l for _, l in val_items], minlength=NUM_CLASSES)
print("")
print("  {:<18}{:>8}{:>10}{:>8}".format("kelas", "latih", "validasi", "porsi"))
for i, nm in enumerate(CLASSES):
    print("  {:<18}{:>8}{:>10}{:>7.1f}%".format(
        nm, n_tr[i], n_va[i], 100.0 * n_va[i] / max(1, n_tr[i] + n_va[i])))

meta = {"seed": CFG["seed"],
        "val_frac": (1.0 / CFG["kfolds"] if CFG["kfolds"] else CFG["val_frac"]),
        "kfolds": CFG["kfolds"], "fold": CFG["fold"] if CFG["kfolds"] else 0,
        "n_val": len(val_items), "limit": CFG["limit"],
        "files": sorted(Path(p).name for p, _ in val_items)}
if CFG["limit"]:
    # CFG["limit"] > 0 -> split validasi berasal dari data yang SUDAH dipangkas.
    # Menulis my_val/ di sini akan menimpa alat ukur 413 gambar dengan subset kecil
    # (sebagian gambarnya juga ikut jadi data latih). Jadi: jangan tulis.
    print("")
    print("[split] CFG['limit'] = {} -> my_val/ TIDAK ditulis (uji cepat saja).".format(
        CFG["limit"]))
    print("        Untuk membangun my_val/ yang benar: set CFG['limit'] = 0, "
          "jalankan ulang sel ini.")
else:
    tulis_my_val(val_items, Path("./my_val"), meta)

# ---------- arsitektur kandidat (kalau CFG["arsitektur"] diisi) ----------
# Berkas kandidat taruh di experiments/. Isinya get_model() + get_transform().
# Kalau tidak diisi, dipakai submission/model.py seperti biasa.
arsitektur = None
PEMUAT_BOBOT = None          # fungsi penyalin bobot (timm atau torchvision)
if CFG.get("arsitektur"):
    import importlib.util

    # Jalur relatif dihitung dari folder kerja (cwd). Di Colab, cwd biasanya
    # /content sedangkan repo hasil clone ada di /content/repo. Jadi kalau tidak
    # ketemu di cwd, dicari juga di REPO_DIR - supaya tidak gagal hanya karena
    # beda folder kerja.
    _p_ar = Path(CFG["arsitektur"])
    if not _p_ar.is_file() and not _p_ar.is_absolute():
        _p_alt = Path(REPO_DIR) / CFG["arsitektur"]
        if _p_alt.is_file():
            print("[arsitektur] '{}' tidak ada di folder kerja, dipakai: {}".format(
                CFG["arsitektur"], _p_alt))
            _p_ar = _p_alt
    if not _p_ar.is_file():
        _dir_exp = Path(REPO_DIR) / "experiments"
        _ada = sorted(p.name for p in _dir_exp.glob("*_model.py")) if _dir_exp.is_dir() else []
        raise SystemExit(
            "[GAGAL] berkas arsitektur tidak ada: {}\n"
            "  folder kerja (cwd)     : {}\n"
            "  folder repo (REPO_DIR) : {}\n"
            "  yang dicoba            : {}\n"
            "  kandidat yang tersedia : {}\n"
            "  Isi CFG[\"arsitektur\"] = \"experiments/<nama berkas>\" (relatif ke repo),\n"
            "  atau tulis jalur lengkapnya.".format(
                CFG["arsitektur"], Path.cwd(), REPO_DIR,
                Path(CFG["arsitektur"]).resolve(),
                _ada if _ada else "(tidak ada berkas *_model.py di experiments/)"))
    _spec = importlib.util.spec_from_file_location("arsitektur_kandidat", _p_ar)
    arsitektur = importlib.util.module_from_spec(_spec)
    try:
        _spec.loader.exec_module(arsitektur)
    except Exception as _e:
        raise SystemExit("[GAGAL] {} gagal dimuat: {}: {}".format(
            _p_ar, type(_e).__name__, _e))
    for _f in ("get_model", "get_transform"):
        if not hasattr(arsitektur, _f):
            raise SystemExit("[GAGAL] {} tidak punya fungsi {}()".format(_p_ar, _f))
    get_model = arsitektur.get_model
    if getattr(arsitektur, "BACKBONE", None):
        CFG["backbone"] = arsitektur.BACKBONE
    PEMUAT_BOBOT = ambil_pemuat_bobot(arsitektur)
    _punya_pemuat = getattr(arsitektur, "muat_bobot_pretrained", None) is not None
    print("[arsitektur] KANDIDAT dari {} (backbone {})".format(_p_ar, CFG["backbone"]))
    print("[bobot] pemuat: {}".format(
        "muat_bobot_pretrained() milik berkas arsitektur (timm)" if _punya_pemuat
        else "torchvision IMAGENET1K_V1"))
else:
    PEMUAT_BOBOT = ambil_pemuat_bobot(None)
    print("[arsitektur] bawaan: submission/model.py")

# Kalau memakai arsitektur kandidat (experiments/...), transform validasinya
# diambil dari kandidat itu - jangan dipaksa 224. Resolusi latih WAJIB mengikuti
# resolusi get_transform(), jadi augmentasi latih disesuaikan otomatis.
# (mis. EfficientNet V2-S memakai 384, bukan 224.)
_gt = arsitektur.get_transform if arsitektur else get_transform
val_tf = _gt()

_kosong = val_tf(Image.new("RGB", (320, 320)))
IMG_LATIH = int(_kosong.shape[-2])
_H = int(_kosong.shape[-1])
if IMG_LATIH != _H:
    raise SystemExit("[GAGAL] get_transform() menghasilkan {}x{} (tidak persegi).".format(IMG_LATIH, _H))
if IMG_LATIH != IMG_SIZE:
    print("[arsitektur] resolusi {}x{} (bukan {}) -> augmentasi latih disesuaikan".format(
        IMG_LATIH, IMG_LATIH, IMG_SIZE))

# Normalisasi ikut berkas arsitektur (EVA-02 memakai normalisasi CLIP). Kalau
# berkas tidak menyediakan MEAN/STD, bawaan ImageNet dipakai - sama seperti train.py.
NORM_MEAN = getattr(arsitektur, "MEAN", None) if arsitektur else None
NORM_STD = getattr(arsitektur, "STD", None) if arsitektur else None
if NORM_MEAN is not None:
    print("[arsitektur] normalisasi dari berkas: mean {} std {}".format(
        tuple(NORM_MEAN), tuple(NORM_STD)))

tr_tf = build_train_transform(IMG_LATIH, NORM_MEAN, NORM_STD)
tr_ds = PathListDataset(train_items, tr_tf)

print("")
print("[kontrak] get_transform() -> {}x{}  (cocok dengan resolusi latih {}x{}) OK".format(
    _H, _H, IMG_LATIH, IMG_LATIH))

va_ds = PathListDataset(val_items, val_tf)

# ---------- PENGAMAN MEMORI: batch otomatis diturunkan bila resolusi besar ----------
# Kebutuhan memori naik ~kuadrat resolusi. Pada 384 (EfficientNet V2-S) batch 32
# TIDAK muat di T4 16 GB -> CUDA out of memory di tengah epoch.
# Di sini batch diturunkan otomatis supaya tidak perlu diingat-ingat.
_batas_batch = {224: 32, 256: 24, 288: 16, 320: 12, 384: 8, 448: 6, 512: 4}
BATCH_AMAN = next((max(1, b) for r, b in sorted(_batas_batch.items()) if IMG_LATIH <= r), 4)
if CFG["batch_size"] > BATCH_AMAN:
    print("[memori] resolusi {}x{} -> batch diturunkan otomatis: {} -> {}".format(
        IMG_LATIH, IMG_LATIH, CFG["batch_size"], BATCH_AMAN))
    CFG["batch_size"] = BATCH_AMAN
else:
    print("[memori] batch {} aman untuk resolusi {}x{}".format(
        CFG["batch_size"], IMG_LATIH, IMG_LATIH))

# drop_last=True: buang batch terakhir yang tidak penuh, supaya statistik
# BatchNorm tidak goyah karena batch kecil (mis. sisa 4 gambar).
tr_dl = DataLoader(tr_ds, CFG["batch_size"], shuffle=True, drop_last=True,
                   num_workers=CFG["num_workers"], pin_memory=True)
va_dl = DataLoader(va_ds, CFG["batch_size"], shuffle=False,
                   num_workers=CFG["num_workers"], pin_memory=True)
print("batch: {} latih, {} validasi".format(len(tr_dl), len(va_dl)))

## 14. LATIH

Sel ini melakukan hal yang sama persis dengan `python train.py` di laptop: susun model,
salin bobot ImageNet, hitung bobot kelas, optimalkan dengan AdamW + warmup/cosine,
pantau **macro-F1** tiap epoch, simpan bobot terbaik, dan berhenti lebih awal bila tidak
membaik.

Perkiraan waktu di T4: **±10–20 menit** untuk 30 epoch (sering berhenti lebih awal).


In [ ]:
# ============================================================
# LATIH
# ============================================================
import json
import time
from pathlib import Path

device = get_device()
seed_everything(CFG["seed"])
# ---------- ke mana bobot disimpan ----------
# Percobaan (bawaan)  : runs/<tag>.pth  -> tiap percobaan punya berkas SENDIRI,
#                       jadi tidak saling menimpa dan submission tetap aman.
# Putaran Final       : submission/model.pth (yang dinilai asisten) -> set
#                       CFG["tulis_submission"] = True.
run_dir = Path("runs") / CFG["tag"]
if CFG["tulis_submission"]:
    out_path = Path("./submission/model.pth")
    print("[submission] DIISI: bobot akan ditulis ke submission/model.pth (putaran Final)")
else:
    out_path = run_dir / (CFG["tag"] + ".pth")
    print("[submission] tidak disentuh. Bobot percobaan ini:", out_path)
    print("             (tiap percobaan punya berkas sendiri - tidak saling menimpa)")
run_dir.mkdir(parents=True, exist_ok=True)

print("=" * 68)
print("  PELATIHAN STUDIO 2 (COLAB)")
print("=" * 68)
print("  perangkat   :", device)
print("  backbone    :", CFG["backbone"],
      "(TANPA bobot pretrained)" if CFG["scratch"] else "(pretrained ImageNet)")
print("  epoch maks  : {}  |  lr {}  |  warmup {}  |  clip-grad {}".format(
    CFG["epochs"], CFG["lr"], CFG["warmup"], CFG.get("clip_grad", 0.0)))
print("  batch       : {}  |  weight decay {}".format(
    CFG["batch_size"], CFG["weight_decay"]))
print("  split       : {:.0%} validasi (dihitung dari seed {})".format(
    CFG["val_frac"], CFG["seed"])
      + ("  -> K-fold {}, fold {}".format(CFG["kfolds"], CFG["fold"]) if CFG["kfolds"] else ""))
print("  hasil       :", run_dir)

# ---------- PERIKSA & BERSIHKAN MEMORI GPU DULU (SEBELUM model dibuat) ----------
# Penting: kalau sel ini dijalankan ULANG tanpa "Restart session", memori GPU
# dari percobaan sebelumnya masih terpakai (bisa 14 GB penuh), sehingga
# model.to(device) langsung gagal. Di sini diperiksa lebih dulu, dengan pesan
# yang jelas - bukan pesan OOM yang membingungkan.
if device.type == "cuda":
    # Buang dulu model/optimizer LAMA dari sel sebelumnya (kalau sel ini
    # dijalankan ulang). Ini sering menghindarkan kebutuhan restart runtime.
    import gc
    _dibuang = []
    # CATATAN: "arsitektur" TIDAK dibuang -- sel ini memakainya lewat PEMUAT_BOBOT.
    # Yang dibuang hanya objek besar (model/optimizer).
    for _nama in ("model", "opt", "info_salin"):
        if _nama in globals():
            del globals()[_nama]
            _dibuang.append(_nama)
    if _dibuang:
        print("[memori] variabel lama dibuang:", ", ".join(_dibuang))
    gc.collect()
    torch.cuda.empty_cache()
    _bebas, _total = torch.cuda.mem_get_info()
    print("[memori] GPU bebas {:.2f} GB dari {:.2f} GB".format(_bebas / 1e9, _total / 1e9))
    if _bebas < 4e9:
        raise SystemExit(
            "[GAGAL] memori GPU masih terpakai (hanya {:.2f} GB bebas dari {:.2f} GB).\n"
            "  Penyebab paling umum: sel ini dijalankan ulang TANPA restart runtime,\n"
            "  sehingga memori percobaan sebelumnya belum dibebaskan.\n"
            "  SOLUSI: menu Runtime -> Restart session, lalu jalankan lagi dari awal\n"
            "  (dataset & repo akan diambil ulang otomatis).".format(
                _bebas / 1e9, _total / 1e9))

model = get_model()
info_salin = None
if not CFG["scratch"]:
    # Pemuat fleksibel: berkas arsitektur timm (EVA-02) menyediakan
    # muat_bobot_pretrained(); backbone torchvision memakai jalur bawaan.
    # get_model() sendiri TIDAK mengunduh apa pun -- hanya membangun bentuk.
    info_salin = PEMUAT_BOBOT(model)
    print("")
    print("[bobot] {} tensor disalin dari ImageNet, {} dilewati (termasuk kepala)".format(
        info_salin["disalin"], len(info_salin["dilewati"])))
model.to(device)
print("[memori] model sudah di GPU ({:,} parameter)".format(
    sum(p.numel() for p in model.parameters())))

if CFG["freeze_epochs"]:
    set_backbone_trainable(model, False)
    print("[strategi] linear probing: backbone dibekukan {} epoch, lalu dibuka penuh".format(
        CFG["freeze_epochs"]))

if CFG["class_weight"]:
    w, counts = class_weight_tensor([l for _, l in train_items], device)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smoothing"])
    print("")
    print("[bobot kelas] (N / (C x n_c), dihitung HANYA dari data latih)")
    for i, nm in enumerate(CLASSES):
        print("  {:<18}{:>6} gambar  ->  bobot {:.3f}".format(nm, int(counts[i]), w[i].item()))
    bobot_freq = float((counts / counts.sum() * w.cpu().numpy()).sum())
    print("  {:<34}= {:.6f}  (selalu 1,0)".format("rata-rata tertimbang frekuensi", bobot_freq))
    print("  {:<34}= {:.6f}  (bukan 1,0 karena kelas tidak seimbang)".format(
        "rata-rata biasa (info saja)", w.mean().item()))
else:
    loss_fn = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])
    print("")
    print("[bobot kelas] dimatikan")

latih = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(latih, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
print("")
print("[optimizer] AdamW, {:,} parameter dilatih".format(sum(p.numel() for p in latih)))

start_epoch = CFG["start_epoch"]
if start_epoch >= CFG["epochs"]:
    start_epoch = max(1, CFG["epochs"] // 2)
    print("[peringatan] start_epoch >= epochs -> diturunkan ke", start_epoch)

best_f1, best_ep, best_acc, no_improve, tersimpan = -1.0, 0, 0.0, 0, False
history = []
print("")
print("-" * 68)

for ep in range(1, CFG["epochs"] + 1):
    lr_now = lr_schedule(ep, CFG["warmup"], CFG["lr"], CFG["eta_min"], CFG["epochs"])

    if CFG["freeze_epochs"] and ep == CFG["freeze_epochs"] + 1:
        # Backbone dibuka -> WAJIB dimasukkan ke optimizer, kalau tidak parameter
        # baru tidak akan pernah ikut diperbarui (gradiennya dibuang).
        baru = set_backbone_trainable(model, True)
        if baru:
            opt.add_param_group({"params": baru, "lr": lr_now,
                                 "weight_decay": CFG["weight_decay"]})
        total = sum(p.numel() for g in opt.param_groups for p in g["params"])
        print("  [strategi] epoch {}: seluruh backbone dibuka -> {} tensor masuk "
              "optimizer (total {:,} parameter dilatih)".format(ep, len(baru), total))

    for g in opt.param_groups:
        g["lr"] = lr_now

    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, tr_dl, loss_fn, opt, device,
                                      clip_grad=CFG.get("clip_grad", 0.0))
    m = evaluate(model, va_dl, loss_fn, device)
    history.append({"epoch": ep, "lr": lr_now, "train_loss": tr_loss, "train_acc": tr_acc,
                    "val_loss": m["loss"], "val_acc": m["acc"], "val_macro_f1": m["macro_f1"]})

    tanda = ""
    # SIMPAN model terbaik sejak epoch 1 (tanpa ambang min_delta).
    if m["macro_f1"] > best_f1:
        best_f1, best_ep, best_acc = m["macro_f1"], ep, m["acc"]
        no_improve = 0
        out_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), out_path)
        tersimpan = True
        tanda = "  <-- TERBAIK, disimpan"
    elif ep >= start_epoch:
        # early stopping baru dihitung setelah pemanasan epoch awal
        if m["macro_f1"] > best_f1 + CFG["min_delta"]:
            no_improve = 0
        else:
            no_improve += 1

    print("  epoch {:2d}/{} | train {:.4f}/{:.4f} | val {:.4f}/{:.4f} | "
          "macro-F1 {:.4f} | lr {:.2e} | {:.1f}s{}".format(
              ep, CFG["epochs"], tr_loss, tr_acc, m["loss"], m["acc"],
              m["macro_f1"], lr_now, time.time() - t0, tanda))

    if ep >= start_epoch and no_improve >= CFG["patience"]:
        print("  -> EARLY STOPPING di epoch {} (macro-F1 tidak membaik {} epoch)".format(
            ep, CFG["patience"]))
        break

print("-" * 68)

if not tersimpan:
    print("[peringatan] tidak ada epoch yang tersimpan -> menyimpan epoch terakhir")
    torch.save(model.state_dict(), out_path)
    best_ep = len(history)

model.load_state_dict(torch.load(out_path, map_location=device))
akhir = evaluate(model, va_dl, loss_fn, device)
print("")
print("HASIL TERBAIK (epoch {})".format(best_ep))
print("  Macro-F1 : {:.4f}   <-- metrik penilaian".format(akhir["macro_f1"]))
print("  Akurasi  : {:.4f}".format(akhir["acc"]))
print("  F1 per kelas:")
for i, nm in enumerate(CLASSES):
    print("    {:<18}{:.4f}".format(nm, akhir["per_class_f1"][i]))
print("  Confusion matrix (baris = asli, kolom = prediksi):")
for i, nm in enumerate(CLASSES):
    print("    {:<18}{}".format(nm, akhir["cm"][i].tolist()))

laporan = {"backbone": CFG["backbone"], "pretrained": not CFG["scratch"],
           "epochs_dijalankan": len(history), "tag": CFG["tag"],
           "best_epoch": best_ep, "val_macro_f1": akhir["macro_f1"],
           "val_acc": akhir["acc"],
           "per_class_f1": dict(zip(CLASSES, akhir["per_class_f1"])),
           "confusion_matrix": akhir["cm"].tolist(), "config": CFG}
if info_salin:
    laporan["salin_bobot"] = {"disalin": info_salin["disalin"],
                              "dilewati": len(info_salin["dilewati"])}

(run_dir / "history.json").write_text(json.dumps(history, indent=2))
(run_dir / "metrics.json").write_text(json.dumps(laporan, indent=2))
(run_dir / "config.json").write_text(json.dumps(CFG, indent=2, default=str))

print("")
print("Bobot :", out_path)
print("Bukti :", run_dir, "/(config|history|metrics).json")

## 15. Uji dengan `evaluate.py` Resmi (wajib)

Sel ini menjalankan **skrip evaluasi milik asisten** pada `my_val/`. Kalau sel ini gagal,
submission belum memenuhi kontrak — dan asisten juga akan gagal menjalankannya.

Angka di sini harus **sama** dengan macro-F1 pada sel 14.


In [ ]:
# Uji bobot HASIL PERCOBAAN INI dengan evaluate.py resmi (seperti asisten).
# Bobotnya diambil dari out_path (dari sel LATIH), bukan dari submission/.
import shutil
import subprocess
import sys
from pathlib import Path

BOBOT_UJI = Path(out_path)
if not BOBOT_UJI.is_file():
    raise SystemExit(
        "[GAGAL] bobot tidak ada: {}\n"
        "  Jalankan sel LATIH dulu (sel 14).".format(BOBOT_UJI))
print("menguji bobot:", BOBOT_UJI, "({:.1f} MB)".format(BOBOT_UJI.stat().st_size / 1e6))

JALAN = Path(REPO_DIR)
if not (JALAN / "evaluate.py").is_file():
    raise SystemExit(
        "[GAGAL] evaluate.py tidak ada di {}\n"
        "  Salin berkas itu dari repo kelompok ke dalam ZIP dataset, karena skrip\n"
        "  inilah yang dipakai asisten dosen untuk menilai submission.".format(JALAN))
if not (JALAN / "submission" / "model.py").is_file():
    raise SystemExit("[GAGAL] submission/model.py tidak ada di {}".format(JALAN))

(JALAN / "submission").mkdir(parents=True, exist_ok=True)
shutil.copy2(BOBOT_UJI, JALAN / "submission" / "model.pth")
if (JALAN / "my_val").exists():
    shutil.rmtree(JALAN / "my_val")
shutil.copytree("./my_val", JALAN / "my_val")

print("menjalankan evaluate.py di", JALAN)
print("")
subprocess.run([sys.executable, "evaluate.py", "--submission", "./submission",
                "--data", "./my_val"], cwd=str(JALAN), check=False)

## 16. Grafik Kurva Pelatihan

Membantu membaca: apakah model masih belajar, sudah mulai menghafal, atau early
stopping-nya bekerja.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

hist = json.loads((Path("runs") / CFG["tag"] / "history.json").read_text())
ep = [r["epoch"] for r in hist]
best = int(np.argmax([r["val_macro_f1"] for r in hist])) + 1

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for a, kunci_tr, kunci_va, judul in [(ax[0], "train_loss", "val_loss", "Loss"),
                                     (ax[1], "train_acc", "val_acc", "Akurasi")]:
    a.plot(ep, [r[kunci_tr] for r in hist], "o-", label="latih")
    a.plot(ep, [r[kunci_va] for r in hist], "s--", label="validasi")
    a.axvline(best, color="crimson", ls=":", label="terbaik (epoch {})".format(best))
    a.set_title(judul)
    a.set_xlabel("epoch")
    a.legend()
    a.grid(alpha=0.3)

ax[2].plot(ep, [r["val_macro_f1"] for r in hist], "o-", color="#1151FF")
ax[2].axvline(best, color="crimson", ls=":", label="terbaik (epoch {})".format(best))
ax[2].set_title("Macro-F1 validasi  (metrik penilaian)")
ax[2].set_xlabel("epoch")
ax[2].legend()
ax[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Path("runs") / CFG["tag"] / "kurva_pelatihan.png",
            dpi=150, bbox_inches="tight")
plt.show()

gap = hist[best - 1]["train_acc"] - hist[best - 1]["val_acc"]
print("epoch terbaik         : {} dari {} dijalankan".format(best, len(hist)))
print("macro-F1 terbaik      : {:.4f}".format(hist[best - 1]["val_macro_f1"]))
print("selisih akurasi tr-val: {:+.4f}".format(gap))
print("  selisih besar -> mulai menghafal (overfitting)")
print("  epoch terbaik jauh sebelum akhir -> early stopping bekerja")

## 17. Confusion Matrix

Lihat kelas mana yang paling sering tertukar. Dari situ bisa disimpulkan kelas mana yang
perlu perhatian tambahan — biasanya `kertas_kardus` karena jumlahnya paling sedikit.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

met = json.loads((Path("runs") / CFG["tag"] / "metrics.json").read_text())
cm = np.array(met["confusion_matrix"], dtype=float)
normal = cm / np.clip(cm.sum(1, keepdims=True), 1, None)

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
# catatan: gambar KIRI memakai nilai BULAT (int) supaya angka yang dicetak tidak
# menampilkan koma (format "d" tidak menerima float). Gambar KANAN tetap float
# karena menampilkan porsi (2 angka di belakang koma).
for a, data, judul, fmt in [(ax[0], cm.astype(int), "jumlah gambar", "d"),
                            (ax[1], normal, "porsi per baris", ".2f")]:
    im = a.imshow(data, cmap="Blues")
    a.set_xticks(range(5))
    a.set_yticks(range(5))
    a.set_xticklabels(CLASSES, rotation=35, ha="right", fontsize=9)
    a.set_yticklabels(CLASSES, fontsize=9)
    a.set_xlabel("prediksi")
    a.set_ylabel("label asli")
    a.set_title(judul)
    for i in range(5):
        for j in range(5):
            warna = "white" if data[i, j] > 0.55 * data.max() else "black"
            a.text(j, i, format(data[i, j], fmt), ha="center", va="center",
                   color=warna, fontsize=8)
    plt.colorbar(im, ax=a, fraction=0.046)

plt.tight_layout()
plt.show()

print("F1 per kelas:")
for nm in CLASSES:
    print("  {:<18}{:.4f}".format(nm, met["per_class_f1"][nm]))
terlemah = min(met["per_class_f1"], key=met["per_class_f1"].get)
print("")
print("kelas terlemah: {} (F1 {:.4f}) — periksa barisnya di gambar kiri".format(
    terlemah, met["per_class_f1"][terlemah]))

## 18. Unduh Hasil

Membuat satu ZIP berisi bobot percobaan ini, `submission/`, `evaluate.py`, dan folder
`runs/` (bukti semua percobaan), lalu mengunduhnya.

**Setelah diunduh:** simpan ZIP-nya di laptop. Untuk putaran Final, bobot yang menang
baru disalin ke `submission/model.pth` lalu di-commit. **Jangan** sertakan dataset atau
`my_val/`.

In [ ]:
import shutil
from pathlib import Path

KELUARAN = Path("/content/hasil_studio2")
if KELUARAN.exists():
    shutil.rmtree(KELUARAN)

# Berkas yang sudah pasti: model.py, requirements.txt, evaluate.py
(KELUARAN / "submission").mkdir(parents=True)
for f in ["model.py", "requirements.txt"]:
    p = Path(REPO_DIR) / "submission" / f
    if p.is_file():
        shutil.copy2(p, KELUARAN / "submission" / f)
shutil.copy2(Path(REPO_DIR) / "evaluate.py", KELUARAN / "evaluate.py")

# Bobot hasil percobaan ini (dari out_path); ke submission/ juga kalau Final
# EVA-02 Base fp32 = 348 MB -> dikonversi ke fp16 (174 MB) supaya unduhannya wajar.
# Prediksi fp16 identik dengan fp32 (diuji pada EVA-02 Small: 0 dari 413 berbeda),
# dan batas submission 500 MB tetap aman.
if Path(out_path).is_file():
    _sumber = Path(out_path)
    _tujuan = KELUARAN / "bobot_percobaan.pth"
    if _sumber.stat().st_size > 120e6:
        import torch as _torch
        _sd = _torch.load(_sumber, map_location="cpu", weights_only=True)
        _sd16 = {k: (v.half() if v.is_floating_point() else v) for k, v in _sd.items()}
        _torch.save(_sd16, _tujuan)
        print("bobot percobaan : {} ({:.1f} MB fp32) -> bobot_percobaan.pth "
              "({:.1f} MB fp16)".format(
                  _sumber, _sumber.stat().st_size / 1e6, _tujuan.stat().st_size / 1e6))
    else:
        shutil.copy2(_sumber, _tujuan)
        print("bobot percobaan :", _sumber, "->", "bobot_percobaan.pth")
if Path("./submission/model.pth").is_file():
    shutil.copy2("./submission/model.pth", KELUARAN / "submission" / "model.pth")

# Bukti tiap percobaan: metrics.json + bobotnya (kalau disimpan di runs/)
shutil.copytree("runs", KELUARAN / "runs", dirs_exist_ok=True)

baris = "| {} | seed {} | split {:.0%} | {:.4f} | {:.4f} | {} |".format(
    CFG["tag"], CFG["seed"], CFG["val_frac"],
    met["val_macro_f1"], met["val_acc"], met["best_epoch"])

teks = ["Hasil pelatihan di Colab", "========================",
        "backbone         : " + CFG["backbone"],
        "pretrained       : {}".format(not CFG["scratch"]),
        "split            : {:.0%} validasi (berstrata), seed {}".format(
            CFG["val_frac"], CFG["seed"]),
        "epoch dijalankan : {}".format(len(hist)),
        "epoch terbaik    : {}".format(met["best_epoch"]),
        "Macro-F1 lokal   : {:.4f}".format(met["val_macro_f1"]),
        "Akurasi lokal    : {:.4f}".format(met["val_acc"]), ""]
teks += ["F1 {:<16}: {:.4f}".format(k, v) for k, v in met["per_class_f1"].items()]
teks += ["", "Baris untuk tabel Hasil di README:", baris]
(KELUARAN / "ringkasan_hasil.txt").write_text("\n".join(teks) + "\n")

shutil.make_archive("/content/hasil_studio2", "zip", KELUARAN)
print("siap: /content/hasil_studio2.zip  ({:.1f} MB)".format(
    Path("/content/hasil_studio2.zip").stat().st_size / 1e6))
print("")
print((KELUARAN / "ringkasan_hasil.txt").read_text())

from google.colab import files
files.download("/content/hasil_studio2.zip")

## 19. Putaran Final — latih **100% data** (lewat `train.py`)

Notebook selalu menyisakan 15% data untuk validasi dan **tidak punya mode `--full-data`**.
Untuk putaran Final, mode itu dijalankan dari sel ini: memanggil **`train.py` di dalam repo**
(kode yang sama persis dengan laptop, fitur lengkap) pada **seluruh 2.753 gambar** — tanpa
validasi, karena tidak ada lagi data uji lokal.

Aturan: **latih ulang di 100% data hanya setelah kandidat MENANG pada split.** Angka
pembandingnya tetap dari run split (`runs/<tag>/metrics.json`), bukan dari run ini.
Isi `EPOCH_FINAL` dengan jumlah epoch run split (mis. 15; kalau early stop di epoch 11 → 11).


In [ ]:
# Putaran Final: latih ulang 100% data (tanpa validasi) memakai train.py di repo.
import subprocess
import sys
from pathlib import Path

EPOCH_FINAL = 15          # <- samakan dengan jumlah epoch run split
TAG_FINAL = "eva02_base_448_lr2e-5_full_seed42"

_jalan = Path(REPO_DIR)
if not (_jalan / "train.py").is_file():
    raise SystemExit("[GAGAL] train.py tidak ada di {} - pull repo dulu.".format(_jalan))

_perintah = [
    sys.executable, "train.py",
    "--data", str(DATA_DIR),
    "--arsitektur", "experiments/eva02_base_448_model.py",
    "--full-data",
    "--epochs", str(EPOCH_FINAL),
    "--lr", "2e-5", "--weight-decay", "0.05", "--warmup", "1", "--clip-grad", "1.0",
    "--batch-size", "6", "--tag", TAG_FINAL,
]
print("perintah:", " ".join(_perintah))
print("")
# --full-data TANPA --to-submission -> submission/ tidak tersentuh.
# Bobot masuk runs/<tag>/<tag>.pth; promosi lewat promote.py di laptop.
subprocess.run(_perintah, cwd=str(_jalan), check=False)

_bobot = _jalan / "runs" / TAG_FINAL / (TAG_FINAL + ".pth")
if _bobot.is_file():
    print("")
    print("bobot final : {} ({:.1f} MB fp32)".format(_bobot, _bobot.stat().st_size / 1e6))
    print("lanjutkan   : unduh bobot ini, lalu promote.py di laptop "
          "(caranya di experiments/README.md)")


## 20. Opsional — K-fold 5 (cara paling hemat memilih konfigurasi)

**Kenapa ini bagus?** Split tunggal membuang sebagian data dari latihan. K-fold 5 **tidak
membuang apa pun**: setiap gambar jadi data latih di 4 dari 5 putaran, dan jadi data
validasi di 1 putaran. Jadi total paparan latihnya **lebih banyak**, sambil angkanya tetap
bisa dipercaya (ambang beda turun ke ±4,2%, dibanding ±4,8% pada split 15%).

**Biaya:** 5× waktu (±35–50 menit per konfigurasi).

Cara pakai: ubah `kfolds` jadi `5` di sel 5, jalankan ulang sel 13–14 untuk `fold` = 1
sampai 5, lalu jalankan sel ini.

> **Penting:** setelah selesai, jalankan **ulang sel 13–14 dengan `kfolds = 0`** supaya
> `submission/model.pth` dan `my_val/` kembali dari konfigurasi penyelesaian terakhir.

In [ ]:
# RATA-RATA HASIL K-FOLD — jalankan setelah semua fold selesai
import glob
import json
from pathlib import Path

berkas = sorted(glob.glob("runs/*kf*/metrics.json"))
if not berkas:
    print("Belum ada hasil K-fold. Jalankan sel 13-14 untuk fold 1..5 dulu.")
else:
    print("{:<28}{:>10}{:>10}{:>7}".format("tag", "macro-F1", "akurasi", "epoch"))
    print("-" * 57)
    f1s = []
    for b in berkas:
        d = json.loads(Path(b).read_text())
        f1s.append(d["val_macro_f1"])
        print("{:<28}{:>10.4f}{:>10.4f}{:>7}".format(
            d["tag"], d["val_macro_f1"], d["val_acc"], d["best_epoch"]))
    mean = sum(f1s) / len(f1s)
    std = (sum((x - mean) ** 2 for x in f1s) / max(1, len(f1s) - 1)) ** 0.5
    print("-" * 57)
    print("{:<28}{:>10.4f}".format("RATA-RATA", mean))
    print("{:<28}{:>10.4f}".format("simpangan baku", std))
    print("")
    print("laporkan di README sebagai: {:.4f} +- {:.4f}".format(mean, std))
    print("(pakai mean dan std ini, bukan angka satu lipatan)")


## 21. Bandingkan Semua Percobaan

Membaca seluruh `runs/<tag>/metrics.json` dan mengurutkannya dari yang terbaik. Dari sini terlihat mana yang menang, dan selisihnya apakah **nyata** atau cuma keberuntungan (ambang beda nyata ±2,7% pada 413 gambar).

> Percobaan yang pakai `limit` (mis. 500 gambar) sengaja diberi tanda — angkanya **tidak sebanding** dengan yang latih penuh.


In [ ]:
# BANDINGKAN SEMUA PERCOBAAN — jalankan kapan saja setelah beberapa percobaan.
# Membaca runs/<tag>/metrics.json yang sudah ada, lalu mengurutkannya.
import glob
import json
import re
from pathlib import Path

berkas = sorted(glob.glob("runs/*/metrics.json"))


def _nama_tag(berkas_metrics):
    """Ambil nama folder run (mis. 'coba_resnet34' atau 'resnet18_kf1_seed42')."""
    return Path(berkas_metrics).parent.name


def _itu_kfold(berkas_metrics):
    """True kalau run ini bagian dari K-fold (nama tag-nya memuat 'kf<N>')."""
    return re.search(r"kf[0-9]+", _nama_tag(berkas_metrics)) is not None


# K-fold dipisah supaya tidak tercampur dengan percobaan biasa: angka K-fold
# dihitung pada data lebih banyak per lipatan, jadi tidak sebanding.
biasa = [b for b in berkas if not _itu_kfold(b)]
kfold = [b for b in berkas if _itu_kfold(b)]


def tampilkan(daftar, judul):
    if not daftar:
        print("(" + judul + ": belum ada)")
        return []
    baris = []
    for b in daftar:
        d = json.loads(Path(b).read_text())
        baris.append((d.get("val_macro_f1", 0.0), d))
    baris.sort(key=lambda x: -x[0])
    print("")
    print(judul)
    print("{:<26}{:>10}{:>10}{:>8}{:>8}  {}".format(
        "tag", "macro-F1", "akurasi", "epoch", "gambar", "catatan"))
    print("-" * 88)
    for f1, d in baris:
        cfg = d.get("config", {}) or {}
        catatan = "backbone {}".format(d.get("backbone", "?"))
        if cfg.get("scratch"):
            catatan += " | dari nol"
        if cfg.get("freeze_epochs"):
            catatan += " | probing {} ep".format(cfg["freeze_epochs"])
        if not cfg.get("class_weight", True):
            catatan += " | tanpa bobot kelas"
        if cfg.get("limit"):
            catatan += " | HANYA {} gambar (bukan angka sebanding)".format(cfg["limit"])
        print("{:<26}{:>10.4f}{:>10.4f}{:>8}{:>8}  {}".format(
            d.get("tag", "?"), f1, d.get("val_acc", 0.0),
            d.get("best_epoch", 0), cfg.get("limit", 0) or "penuh", catatan))
    print("-" * 88)
    print("terbaik: {}  (macro-F1 {:.4f})".format(baris[0][1].get("tag"), baris[0][0]))
    return baris


print("=" * 88)
print("  PERBANDINGAN PERCOBAAN")
print("=" * 88)
utama = tampilkan(biasa, "A. Percobaan biasa (split tunggal 85/15)")

if utama:
    print("")
    # Ambang beda nyata: PERKIRAAN KASAR 1,96 x akar(0,25/n), bukan uji statistik
    # formal. Ukuran validasi default (split 85/15) = 413 gambar. Rumusnya sama
    # seperti experiments/promote.py, supaya angka di seluruh repo konsisten.
    N_VAL_DEFAULT = 413
    AMBANG = 1.96 * (0.25 / N_VAL_DEFAULT) ** 0.5
    print("Selisih dengan peringkat 1 (ambang beda nyata +-{:.1f}% pada {} gambar):".format(
        100 * AMBANG, N_VAL_DEFAULT))
    juara = utama[0][0]
    for f1, d in utama[1:]:
        tanda = "  <-- di bawah ambang, anggap SERI" if abs(juara - f1) < AMBANG else ""
        print("  {:<26}{:+.4f}{}".format(d.get("tag"), f1 - juara, tanda))
    print("")
    print("-> Percobaan yang menang:")
    print("   1. CFG['tag'] = '" + utama[0][1].get("tag", "") + "'")
    print("   2. jalankan ULANG sel 13 & 14 dengan CFG itu (supaya bobotnya benar)")
    print("   3. jalankan sel 'Bandingkan lagi' kalau perlu, lalu sel uji evaluate.py")

tampilkan(kfold, "B. K-fold (angka lebih bisa dipercaya, tapi bukan 100% data)")
print("")
print("Catatan: percobaan ber-`limit` (mis. 500 gambar) TIDAK sebanding dengan yang penuh.")
print("Jalankan ulang yang ber-limit dengan data penuh sebelum menyimpulkan apa pun.")

## 22. Opsional — Eksperimen Strategi Fine-Tuning (diminta RPS)

RPS Studio 2 meminta perbandingan **freezing / linear probing / full fine-tuning**.
Jalankan satu per satu, catat macro-F1-nya, lalu laporkan tabelnya di README.

> Setiap percobaan otomatis punya berkasnya sendiri di `runs/<tag>/` (dan bobotnya
> `runs/<tag>.pth`), jadi **tidak saling menimpa**. `submission/` hanya tersentuh kalau
> `tulis_submission = True`. **Selalu isi `tag` berbeda** supaya angkanya mudah dibandingkan.

In [ ]:
# Cara pakai: jalankan baris "CFG[...]" yang diinginkan, lalu jalankan ULANG sel 13 & 14.
# Baca macro-F1 di akhir sel 14, catat, lalu lanjut ke variasi berikutnya.

# (a) LINEAR PROBING murni — backbone dibekukan penuh, hanya kepala dilatih
#     CFG["freeze_epochs"] = CFG["epochs"]; CFG["tag"] = "linear_probing"

# (b) FULL FINE-TUNING — seluruh backbone dilatih (dipakai untuk putaran Final)
#     CFG["freeze_epochs"] = 0; CFG["tag"] = "full_finetune"

# (c) CNN FROM SCRATCH — tanpa bobot ImageNet (pembanding materi Pertemuan 4)
#     CFG["scratch"] = True; CFG["tag"] = "scratch"

# (d) TANPA BOBOT KELAS — ablasi, melihat pengaruh kelas kertas_kardus
#     CFG["class_weight"] = False; CFG["tag"] = "tanpa_bobot_kelas"

# (e) UJI CEPAT — pakai sebagian data saja, untuk memastikan skrip jalan
#     CFG["limit"] = 500; CFG["epochs"] = 3; CFG["tag"] = "uji_cepat"

print("Sel ini hanya catatan. Jalankan baris CFG di atas, lalu sel 13-14.")
print("Setiap variasi menghasilkan runs/<tag>/metrics.json sebagai bukti.")

## Catatan Penutup

### Cara yakin hasil Colab = hasil laptop

Sel kode bagian **6–12** adalah **salinan langsung dari `train.py`** (termasuk
`build_train_transform()` yang meneruskan `MEAN`/`STD` dari berkas arsitektur, dan
`train_one_epoch(..., clip_grad=...)`), dan urutan langkahnya sama: hitung pembagian data →
susun model → salin bobot → hitung bobot kelas → AdamW → loop (warmup+cosine, early stopping
berdasarkan macro-F1) → simpan bobot terbaik.

| Hal | Notebook (sel 5) | Laptop (CLI) |
|---|---|---|
| pembagian | `val_frac = 0.15` | `--val-frac 0.15` |
| seed pembagian | `seed = 42` | `--seed 42` |
| epoch | `epochs = 15` | `--epochs 15` |
| lr / warmup | `lr = 2e-5`, `warmup = 1` | `--lr 2e-5 --warmup 1` |
| weight decay | `weight_decay = 0.05` | `--weight-decay 0.05` |
| clip gradien | `clip_grad = 1.0` | `--clip-grad 1.0` |
| batch | `batch_size = 6` (448 px) | `--batch-size 6` |
| early stopping | `patience 5`, `start_epoch 6` | `--patience 5 --start-epoch 6` |
| bobot kelas | `class_weight = True` | aktif secara bawaan |

### Mode 100% data untuk putaran Final

Notebook selalu menyisakan 15% data untuk validasi dan **tidak punya mode `--full-data`**.
Putaran Final dijalankan lewat **sel 19**, yang memanggil `train.py` di dalam repo:

```bash
python train.py --data "<...>/train" --arsitektur experiments/eva02_base_448_model.py \
  --full-data --epochs 15 --lr 2e-5 --weight-decay 0.05 --warmup 1 --clip-grad 1.0 \
  --batch-size 6 --tag eva02_base_448_lr2e-5_full_seed42
```

`--full-data` **tidak** menyentuh `submission/` (tanpa `--to-submission`); promosi tetap
lewat `experiments/promote.py` di laptop.

### Sebelum commit ke repo

| # | Periksa |
|---|---|
| 1 | `evaluate.py` (sel 15) jalan tanpa error |
| 2 | Macro-F1 sel 15 sama dengan sel 14 |
| 3 | `salin_bobot` di `metrics.json` = {disalin: ±234, dilewati: 2}; kalau jauh lebih sedikit, bobot tidak tersalin |
| 4 | ukuran `model.pth`: EVA-02 Base fp16 ±174 MB → **lewat batas 100 MB GitHub** → wajib GitHub Release / Git LFS + tautan di README (batas submission 500 MB aman). EVA-02 Small fp16 43,5 MB → commit biasa |
| 5 | waktu evaluasi di sel 15 jauh di bawah 15 menit (EVA-02 Small ±8,5 dtk di GPU; EVA-02 Base ±4–5× lebih berat) |
| 6 | commit `submission/model.py` **dan** `submission/model.pth` bersamaan |
| 7 | jangan commit dataset, `my_val/`, bobot `runs/*.pth`, atau `train_sources.csv` — berkas bukti `runs/<tag>/{config,history,metrics}.json` + `kurva_pelatihan.png` IKUT di-commit |

### Kalau kehabisan memori (OOM)

Batch sudah diturunkan otomatis ke 6 pada 448 px. Kalau masih OOM: **Runtime → Restart
session**, lalu jalankan ulang. Untuk uji cepat boleh `CFG["limit"] = 600` (my_val/ aman,
tidak ditulis saat limit aktif).
